# Phase 1 — Preprocessing

## Purpose

This notebook creates model-ready train and test CSV files without fitting a model.
It applies conservative text preprocessing, encodes the target, and uses a
reproducible stratified split.

In [1]:
from pathlib import Path
import os

# This makes the notebook work whether it is opened from the project folder
# or from inside the notebooks folder.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
IMAGES = PROJECT_ROOT / "images"

DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
IMAGES.mkdir(parents=True, exist_ok=True)

# Store the downloaded Hugging Face files inside this project.
os.environ["HF_HOME"] = str(PROJECT_ROOT / ".cache" / "huggingface")

print("Project folder:", PROJECT_ROOT)

Project folder: E:\AssDS\Project


In [2]:
import re

import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv(DATA_PROCESSED / "roman_urdu_cleaned.csv")
print("Input shape:", df.shape)
df.head()

Input shape: (19490, 5)


,sentence,sentiment,clean_text,character_count,word_count
0,Sai kha ya her kisi kay bus ki bat nhi hai lak...,Positive,sai kha ya her kisi kay bus ki bat nhi hai lak...,97,25
1,sahi bt h,Positive,sahi bt h,9,3
2,"Kya bt hai,",Positive,"kya bt hai,",11,3
3,Wah je wah,Positive,wah je wah,10,3
4,Are wha kaya bat hai,Positive,are wha kaya bat hai,20,5


## 1. Prepare model text

We lowercase text, replace web links and usernames with reusable tokens, and make
spaces consistent. We do **not** remove punctuation, stem words, or delete common
words because those choices can remove sentiment information and are especially
risky for informal Roman Urdu.

In [3]:
def prepare_model_text(text):
    # Create a conservative model-ready version of one sentence.
    text = str(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " <url> ", text)
    text = re.sub(r"@\w+", " <user> ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


df["model_text"] = df["sentence"].apply(prepare_model_text)
df[["sentence", "model_text"]].head(10)

,sentence,model_text
0,Sai kha ya her kisi kay bus ki bat nhi hai lak...,sai kha ya her kisi kay bus ki bat nhi hai lak...
1,sahi bt h,sahi bt h
2,"Kya bt hai,","kya bt hai,"
3,Wah je wah,wah je wah
4,Are wha kaya bat hai,are wha kaya bat hai
5,Wah kya baat likhi,wah kya baat likhi
6,Wha Itni sari khubiya,wha itni sari khubiya
7,Itni khubiya,itni khubiya
8,Ya allah rehm farma hm sab pe or zalimo ko hid...,ya allah rehm farma hm sab pe or zalimo ko hid...
9,Please Everyone AllAh S.w.T ka naam hAmesha Ba...,please everyone allah s.w.t ka naam hamesha ba...


## 2. Encode the target labels

In [4]:
# The mapping is written explicitly so it will stay the same in Phase 2.
label_mapping = {"Negative": 0, "Neutral": 1, "Positive": 2}
df["label"] = df["sentiment"].map(label_mapping)

print("Label mapping:", label_mapping)
print("Missing encoded labels:", df["label"].isna().sum())
df[["sentiment", "label"]].drop_duplicates().sort_values("label")

Label mapping: {'Negative': 0, 'Neutral': 1, 'Positive': 2}
Missing encoded labels: 0


,sentiment,label
9497,Negative,0
3631,Neutral,1
0,Positive,2


## 3. Split into train and test sets

We use 80% for training and 20% for testing. `stratify` keeps approximately the
same sentiment percentages in both sets. `random_state=42` makes the split exactly
reproducible. The test set must remain untouched during Phase 2 model training.

In [5]:
train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["label"],
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Training rows:", len(train_df))
print("Test rows:", len(test_df))

Training rows: 15592
Test rows: 3898


In [6]:
def class_percentage(frame):
    return (frame["sentiment"].value_counts(normalize=True) * 100).round(2)


distribution_check = pd.DataFrame(
    {
        "full_data_%": class_percentage(df),
        "train_%": class_percentage(train_df),
        "test_%": class_percentage(test_df),
    }
)
distribution_check

,full_data_%,train_%,test_%
sentiment,,,
Neutral,43.34,43.34,43.33
Positive,29.81,29.81,29.81
Negative,26.85,26.85,26.86


## 4. Why numeric scaling is not applied

The main input is text, not a continuous measurement. StandardScaler or MinMaxScaler
cannot be applied directly to sentences. In Phase 2:

- TF-IDF will create and normalize sparse numeric features for classical models;
- an LSTM/transformer will tokenize text and learn embeddings;
- if hand-built numeric length features are added to a model, a scaler must be fit
  on the training set only and then applied to the test set.

Doing that later prevents data leakage and keeps this Phase 1 output reusable.

## 5. Save processed files and run leakage checks

In [7]:
processed_path = DATA_PROCESSED / "roman_urdu_processed.csv"
train_path = DATA_PROCESSED / "train.csv"
test_path = DATA_PROCESSED / "test.csv"

df.to_csv(processed_path, index=False, encoding="utf-8")
train_df.to_csv(train_path, index=False, encoding="utf-8")
test_df.to_csv(test_path, index=False, encoding="utf-8")

train_texts = set(train_df["model_text"])
test_texts = set(test_df["model_text"])

print("Missing values in processed data:", int(df.isna().sum().sum()))
print("Duplicate model texts:", int(df["model_text"].duplicated().sum()))
print("Texts appearing in both train and test:", len(train_texts.intersection(test_texts)))
print("Saved:", processed_path)
print("Saved:", train_path)
print("Saved:", test_path)

Missing values in processed data: 0
Duplicate model texts: 0
Texts appearing in both train and test: 0
Saved: E:\AssDS\Project\data\processed\roman_urdu_processed.csv
Saved: E:\AssDS\Project\data\processed\train.csv
Saved: E:\AssDS\Project\data\processed\test.csv


## Preprocessing conclusion

The saved files contain readable sentiment names, fixed numeric labels, and a
conservative model-ready text column. The stratified split is reproducible and has
no repeated normalized text shared between train and test.